# 4주차. MCP 서버 Tool Call과 SQLite 저장 흐름 연결하기

## 0. 목표

이번 주에는 Python 함수 tool을 직접 넘기는 방식에서 한 걸음 나아가, 별도 터미널에서 실행 중인 로컬 MCP HTTP 서버가 노출한 tool을 agent가 호출하게 만든다. 일정 생성 tool은 SQLite에 row를 저장한다.

성취 기준:

- MCP를 외부 서버에서 표준 방식으로 tool을 가져오는 방법으로 설명할 수 있다.
- Python 함수 tool과 MCP tool의 차이를 payload 기준으로 말할 수 있다.
- MCP payload의 `event_id`와 SQLite 저장 row의 `event_id`를 비교할 수 있다.

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 실제 OpenAI API를 호출한다. API 키와 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → `week04.py`의 `# TODO 문제`를 먼저 풀고 바로 아래 `# 모범 답안`과 비교한 뒤 Gradio 데모를 실행한다.

4주차는 MCP 서버를 따로 실행한다. 노트북을 실행하기 전에 repo 루트의 다른 터미널에서 아래 명령을 켜 둔다.

```bash
python week04_mcp_server.py
```

기본 MCP endpoint는 `http://127.0.0.1:8004/mcp`이다.


In [ ]:
# 흐름: repo 설정과 공통 helper를 준비한다.
# 4주차 노트북은 실행 전에 별도 터미널에서 week04_mcp_server.py가 켜져 있어야 한다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    # 현재 폴더부터 부모 폴더로 올라가며 repo 루트를 찾는다.
    # 노트북을 notebook/ 안에서 실행해도, repo 루트에서 실행해도 같은 경로를 쓰기 위한 장치다.
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


# 앞으로 나오는 상대 경로는 모두 이 repo 루트를 기준으로 맞춘다.
REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    # weekXX.py 같은 repo 루트의 Python 파일을 import할 수 있게 경로를 추가한다.
    sys.path.insert(0, str(REPO_ROOT))
# 파일 저장/DB 생성 위치가 흔들리지 않도록 작업 폴더를 repo 루트로 고정한다.
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
load_dotenv(REPO_ROOT / ".env", override=True)

# 모델 이름은 .env에서 바꿀 수 있고, 값이 없으면 수업 기본값을 사용한다.
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    # API key가 없으면 모델 호출 오류가 길게 나오므로, 준비 셀에서 바로 멈춘다.
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    # temperature=0은 같은 입력에서 비슷한 tool 선택/구조화 결과가 나오게 하기 위한 설정이다.
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    # ensure_ascii=False로 한글 payload를 사람이 읽기 쉬운 그대로 출력한다.
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    # agent 실행 결과에서 마지막 message가 사용자에게 보이는 최종 답변이다.
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    # LangChain message 전체를 그대로 보여주면 복잡하므로, 수업에서 볼 핵심만 뽑는다.
    trace = []
    for message in agent_result.get("messages", []):
        # tool_calls는 모델이 "이 도구를 이 인자로 실행해줘"라고 요청한 기록이다.
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            # type == "tool"인 message는 실제 tool 실행 결과다.
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: MCP는 도구를 agent 코드 안에 직접 넣는 대신, 외부 서버가 표준 방식으로 tool 목록과 실행 결과를 제공하게 해준다. 이번 주에는 아주 작은 로컬 FastMCP 서버를 별도 터미널에서 실행하고, 노트북과 UI는 HTTP endpoint로 요청한다.

반드시 이해할 것:

- agent는 MCP 서버에서 읽어온 tool을 호출한다.
- 성공 여부는 MCP payload의 `server`, `tool`, `arguments`, `status`로 확인한다.
- SQLite row는 MCP tool이 실제로 로컬 저장소에 기록했다는 근거다.
- MCP를 쓰면 tool 제공자와 agent 실행 코드를 분리할 수 있다.

지금은 몰라도 되는 것:

- `streamable-http` transport 세부 동작
- FastMCP 서버 런타임 내부 구조
- SQLite query optimizer나 파일 잠금 세부
- `MultiServerMCPClient` 구현 세부

막혔을 때 볼 trace: `calendar_create_event` tool call, MCP tool result content, `parse_mcp_tool_result(...)`로 변환한 payload, `load_saved_calendar_events()` 결과.


## 3. 기본 개념 실습

가장 작은 흐름은 이미 실행 중인 로컬 MCP 서버에서 tool 목록을 읽어오는 것이다. 여기서는 `http://127.0.0.1:8004/mcp`에 연결해 `calendar_check_availability` tool을 호출한다.


In [ ]:
# 흐름: 실행 중인 MCP HTTP 서버에서 tool 목록을 읽고, availability tool을 직접 호출한다.
# MCP tool은 async-only라 run_async(tool.ainvoke(...)) 형태로 실행한다.
# MCP tool 호출은 비동기 API라 asyncio와 event loop 처리가 필요하다.
import asyncio
import pathlib
import sqlite3
import threading

from langchain_mcp_adapters.client import MultiServerMCPClient

DEFAULT_WEEK04_DB_PATH = pathlib.Path(
    os.getenv("KANAMATE_WEEK04_DB_PATH", REPO_ROOT / "tmp" / "week04_calendar.sqlite3")
).resolve()
DEFAULT_WEEK04_MCP_HOST = os.getenv("KANAMATE_WEEK04_MCP_HOST", "127.0.0.1")
DEFAULT_WEEK04_MCP_PORT = int(os.getenv("KANAMATE_WEEK04_MCP_PORT", "8004"))
WEEK04_MCP_URL = f"http://{DEFAULT_WEEK04_MCP_HOST}:{DEFAULT_WEEK04_MCP_PORT}/mcp"


def resolve_calendar_db_path(db_path: str | pathlib.Path | None = None) -> pathlib.Path:
    return pathlib.Path(db_path or DEFAULT_WEEK04_DB_PATH).resolve()


def initialize_calendar_db(db_path: str | pathlib.Path | None = None, reset: bool = False) -> pathlib.Path:
    target_path = resolve_calendar_db_path(db_path)
    # tmp 폴더가 없으면 SQLite 파일을 만들 수 없으므로 먼저 만든다.
    target_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(target_path) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS calendar_events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_id TEXT UNIQUE NOT NULL,
                title TEXT NOT NULL,
                date TEXT NOT NULL,
                start_time TEXT NOT NULL,
                members_json TEXT NOT NULL,
                status TEXT NOT NULL
            )
            """
        )
        if reset:
            conn.execute("DELETE FROM calendar_events")
        conn.commit()
    return target_path


def load_saved_calendar_events(db_path: str | pathlib.Path | None = None) -> list[dict[str, Any]]:
    target_path = initialize_calendar_db(db_path)
    with sqlite3.connect(target_path) as conn:
        # Row factory를 쓰면 row["event_id"]처럼 컬럼명으로 읽을 수 있다.
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT event_id, title, date, start_time, members_json, status
            FROM calendar_events
            ORDER BY id
            """
        ).fetchall()
    return [
        {
            "event_id": row["event_id"],
            "title": row["title"],
            "date": row["date"],
            "start_time": row["start_time"],
            "members_json": row["members_json"],
            "members": json.loads(row["members_json"]),
            "status": row["status"],
        }
        for row in rows
    ]


def run_async(coro):
    'Run an async MCP call from both notebooks and plain Python.'
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        # 일반 Python 실행에서는 이미 돌아가는 event loop가 없으므로 asyncio.run을 바로 사용한다.
        return asyncio.run(coro)

    # Jupyter에서는 event loop가 이미 있으므로, 새 thread 결과를 담을 작은 박스를 만든다.
    box: dict[str, Any] = {}

    def runner() -> None:
        try:
            box["value"] = asyncio.run(coro)
        except Exception as exc:
            box["error"] = exc

    # 새 thread에서 async MCP 호출을 끝낸 뒤 원래 셀로 결과를 돌려받는다.
    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


def make_calendar_mcp_client(server_url: str = WEEK04_MCP_URL) -> MultiServerMCPClient:
    # MultiServerMCPClient는 실행 중인 MCP 서버의 tool 목록을 LangChain tool처럼 읽어온다.
    return MultiServerMCPClient(
        {
            "calendar": {
                "url": server_url,
                "transport": "streamable_http",
            }
        }
    )


def load_calendar_mcp_tools(server_url: str = WEEK04_MCP_URL) -> list[Any]:
    client = make_calendar_mcp_client(server_url)
    try:
        # calendar 서버가 노출한 tool들을 실제 agent tools 목록으로 사용할 것이다.
        return run_async(client.get_tools(server_name="calendar"))
    except Exception as exc:
        raise RuntimeError(
            "4주차 MCP 서버에 연결할 수 없습니다. 먼저 다른 터미널에서 "
            f"`python week04_mcp_server.py`를 실행하세요. MCP URL: {server_url}"
        ) from exc


def mcp_tool_by_name(tools: list[Any], name: str):
    for tool_item in tools:
        if tool_item.name == name:
            return tool_item
    raise KeyError(f"MCP tool을 찾을 수 없습니다: {name}")


def parse_mcp_tool_result(content: Any) -> dict[str, Any]:
    'Convert MCP content blocks or JSON strings into a dict payload.'
    if isinstance(content, dict):
        # adapter가 이미 dict로 돌려주면 바로 payload로 사용한다.
        return content
    if isinstance(content, list):
        # content block 리스트라면 text block 안의 JSON을 찾아 읽는다.
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                return json.loads(item["text"])
            if hasattr(item, "text"):
                return json.loads(item.text)
        raise ValueError(f"MCP text content를 찾지 못했습니다: {content}")
    if isinstance(content, str):
        return json.loads(content)
    raise TypeError(f"지원하지 않는 MCP tool result 형식입니다: {type(content)}")


initialize_calendar_db(reset=True)
show_json({"mcp_url": WEEK04_MCP_URL, "server_command": "python week04_mcp_server.py"})
calendar_mcp_tools = load_calendar_mcp_tools()
show_json([{"name": tool_item.name, "description": tool_item.description} for tool_item in calendar_mcp_tools])

check_tool = mcp_tool_by_name(calendar_mcp_tools, "calendar_check_availability")
basic_mcp_response = run_async(check_tool.ainvoke({"members": ["민수", "지아"], "date": "2026-04-24"}))
basic_mcp_payload = parse_mcp_tool_result(basic_mcp_response)
show_json(basic_mcp_payload)

mcp_agent = create_agent(
    model=make_model(600),
    tools=calendar_mcp_tools,
    system_prompt="그룹 가능 시간 질문은 실제 MCP 서버에서 로드한 calendar_check_availability 도구를 호출한 뒤 답한다.",
)

student_request = "민수와 지아가 2026-04-24에 가능한 시간 알려줘"
result = run_async(mcp_agent.ainvoke({"messages": [{"role": "user", "content": student_request}]}))

print(final_text(result))
show_json(extract_tool_trace(result))


## 4. 카나메이트 확장 예제

같은 로컬 MCP 서버가 `calendar_create_event` tool도 제공한다. Agent는 Python 함수가 아니라 MCP 서버에서 읽어온 tool 목록으로 가능 시간 조회와 일정 생성을 처리하고, 생성 tool은 SQLite에 저장 row를 남긴다.


In [ ]:
# 흐름: 같은 MCP 서버에서 일정 생성 tool까지 로드하고 agent에 연결한다.
# 여기서는 DB를 초기화한 뒤 payload와 SQLite row를 비교할 준비를 한다.
initialize_calendar_db(reset=True)
# 실행 중인 week04_mcp_server.py에서 tool 스키마를 가져온다.
practice_mcp_tools = load_calendar_mcp_tools()
show_json([{"name": tool_item.name, "description": tool_item.description} for tool_item in practice_mcp_tools])
show_json({"sqlite_path": str(DEFAULT_WEEK04_DB_PATH), "saved_events_before": load_saved_calendar_events()})

practice_mcp_agent = create_agent(
    model=make_model(700),
    tools=practice_mcp_tools,
    system_prompt="가능 시간 조회는 calendar_check_availability를, 일정 생성이나 확정 요청은 calendar_create_event MCP 도구를 호출한 뒤 답한다.",
)


## 5. 확장 예제 실행

확정된 그룹 일정을 만들어 달라고 요청하고, trace에 남은 실제 MCP 서버 tool result와 SQLite 저장 row를 함께 확인한다.


In [ ]:
# 흐름: agent.ainvoke로 일정 생성 요청을 실행하고, MCP payload와 SQLite row를 검증한다.
# StructuredTool은 sync invoke를 지원하지 않으므로 ainvoke를 유지한다.
practice_request = "민수와 지아의 발표 리허설을 2026-04-24 15:00 일정으로 생성해줘"
practice_result = run_async(practice_mcp_agent.ainvoke({"messages": [{"role": "user", "content": practice_request}]}))
practice_trace = extract_tool_trace(practice_result)
create_event_payloads = [
    parse_mcp_tool_result(event["content"])
    for event in practice_trace
    if event["event"] == "tool_result" and event["tool_name"] == "calendar_create_event"
]
# 일정 생성 요청은 하나이므로 첫 번째 create_event payload를 검증 대상으로 삼는다.
created_event = create_event_payloads[0]
saved_events = load_saved_calendar_events()

print(final_text(practice_result))
show_json(practice_trace)
show_json(created_event)
show_json(saved_events)

assert any(event["event"] == "tool_call" and event["tool_name"] == "calendar_create_event" for event in practice_trace)
assert created_event["server"] == "kanamate-calendar"
assert created_event["tool"] == "calendar.create_event"
assert created_event["status"] == "created"
assert saved_events
assert saved_events[-1]["event_id"] == created_event["event_id"]
assert saved_events[-1]["status"] == created_event["status"]
print("4주차 확장 예제 실행 통과")


## 6. 회고

개념 확인 질문:

1. Python 함수 tool과 MCP tool은 어디에서 tool 목록을 가져온다는 점이 다른가?
2. MCP payload에서 사람이 반드시 확인해야 할 필드는 무엇인가?
3. SQLite 저장 row는 MCP payload와 어떤 값으로 연결되는가?
4. `streamable-http`, `FastMCP`, `MultiServerMCPClient` 중 지금 깊게 몰라도 되는 것은 무엇이며, 그래도 관찰해야 하는 결과는 무엇인가?
5. MCP tool을 agent에서 실행할 때 `invoke`가 아니라 `ainvoke`를 써야 하는 이유는 무엇인가?

작은 응용 과제: 날짜와 멤버를 바꾼 요청을 실행하고 MCP payload의 `arguments`, `event_id`, SQLite row가 어떻게 달라지는지 비교한다.
